In [22]:
import torch
import torch.nn as nn
from spin_lattices import KagomeLattice, SquareLattice1Diag, TriangleLattice
from heisenberg_hamiltonians import HeisenbergJ1J2
from loguru import logger
from slater_determinant import SlaterDeterminant, tight_binding_init, HalfSpaceProjector
from pathlib import Path
import numpy as np
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import torch.nn.utils.parametrize as parametrize
from torch.nn.utils.parametrizations import orthogonal
from fast_boolean_analysis import fourier_expand
from lattice_boolean_analysis import LatticeBooleanFunction, LBFFromEigenstateSeries, LBFFromSpinSystem
import pandas as pd

def sign_overlap(ground_state, predict_signs):
    probs = ground_state**2
    return torch.dot(ground_state, predict_signs * torch.abs(ground_state)) / probs.sum()

In [4]:
lattice = KagomeLattice(2, 4)
system = HeisenbergJ1J2(lattice, J1=1, J2=1, ground_state_cache_dir=Path("groundstates"))
system.get_eigenstates(1)

ground_state_np = np.real_if_close(system.get_ground_state_in_canonical_basis())
ground_state = torch.from_numpy(ground_state_np)

2023-04-26 14:22:10.420 | DEBUG    | heisenberg_hamiltonians:__init__:427 - use_symmetries is None and lattice is in symmetries whitelist, setting use_symmetries=True, spin_inversion=1
2023-04-26 14:22:10.421 | DEBUG    | heisenberg_hamiltonians:__init__:448 - number_spins=24
2023-04-26 14:22:10.428 | DEBUG    | heisenberg_hamiltonians:__init__:458 - Symmetry group contains 16 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-04-26 14:22:10.504 | DEBUG    | heisenberg_hamiltonians:__init__:467 - Hilbert space dimension is 85662
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-04-26 14:22:10.628 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:60 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-1.0-True-1-1.pickle
2023-04-26 14:22:10.684 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:107 - Ground state energy is -42.8245991763
2023-04-26 14:22:10.686 | DEBUG    | heisenberg

In [6]:
# with torch.no_grad():
#     det.f.copy_(
#         nn.Parameter(
#             torch.randn(system.number_spins, system.number_spins, dtype=torch.float64)
#             / np.sqrt(system.number_spins)
#         )
#     )

eps_train = 0.01
test_size = 10000
epochs = 100
batch_size = 64
lr = 1e-3
scaling = 10000000


for run in range(10):
    dataset_seed = run

    np.random.seed(dataset_seed)
    train_set_numpy = np.random.choice(
        len(system.canonical_basis.states),
        int(eps_train * len(system.canonical_basis.states)),
        replace=False,
        p=ground_state**2,
    )

    train_set = torch.from_numpy(train_set_numpy)
    logger.debug(f"{len(train_set)=}")

    target = (ground_state[train_set] > 0).double()

    rest_set_np = np.setdiff1d(np.arange(len(system.canonical_basis.states)), train_set_numpy)
    rest_probs = ground_state_np[rest_set_np] ** 2
    rest_probs /= rest_probs.sum()
    test_set = torch.from_numpy(np.random.choice(rest_set_np, test_size, replace=False))


    logger.debug(f"{run=}")
    writer = SummaryWriter(
        log_dir=(
            f"experiments/{datetime.now().strftime('%Y_%m_%d')}/{datetime.now().strftime('%H_%M_%S')}"
            f"_{eps_train=}_{batch_size=}_{lr=}"
        #    f"_{initialization=}"
        #    f"_{scaling=}_{keep_symmetries=}"
        )
    )

    torch.manual_seed(run)
    det = orthogonal(SlaterDeterminant(
        system.lattice,
        system.canonical_basis,
        initialization='randn',
        sign_cache_dir=Path("signs_cache"),
    ), name="f")

    parametrize.register_parametrization(det, "f", HalfSpaceProjector())

    n_batches = len(train_set) // batch_size

    criterion = nn.BCELoss()

    optimizer = torch.optim.Adam(det.parameters(), lr=lr)

    epoch = 0
    logger.debug(f"{n_batches=}")
    for epoch in range(epochs):  # loop over the dataset multiple times
        i = None
        loss = None


        for i in range(n_batches):
            x = train_set[i * batch_size : (i + 1) * batch_size]
            y = target[i * batch_size : (i + 1) * batch_size]

            # zero the parameter gradients
            optimizer.zero_grad()

            # forward + backward + optimize
            det_output = det(x) * scaling
            outputs = torch.sigmoid(det_output)
            
            loss = criterion(outputs, y)
            loss.backward()

            optimizer.step()

        assert loss is not None

        overlap_train = sign_overlap(ground_state[train_set], torch.sign(det(train_set) * scaling))
        overlap_test = sign_overlap(ground_state[test_set], torch.sign(det(test_set) * scaling))
        overlap_train_unweighted = torch.sign(ground_state[train_set] * det(train_set)).mean()

        writer.add_scalar("Loss/train", loss.item(), epoch)
        writer.add_scalar("Overlap/train", overlap_train.item(), epoch)
        writer.add_scalar("Overlap/test", overlap_test.item(), epoch)
        writer.add_scalar("Overlap/train/unweighted", overlap_train_unweighted.item(), epoch)

        # writer.add_scalar("Det_output/std/train", det_output.std().item(), epoch)

        # log.append(
        #     {
        #         "epoch": epoch,
        #         "loss": loss.item(),
        #         "overlap_train": overlap_train.item(),
        #         "overlap_test": overlap_test.item(),
        #     }
        # )
        # logger.debug(
        #     f"Epoch {epoch} loss: {loss.item():.4f} overlap_train: {overlap_train.item():.4f} "
        #     f"overlap_test: {overlap_test.item():.4f}"
        # )
            

2023-04-26 14:29:41.011 | DEBUG    | __main__:<module>:29 - len(train_set)=27041
2023-04-26 14:29:41.424 | DEBUG    | __main__:<module>:39 - run=0
2023-04-26 14:29:43.822 | DEBUG    | slater_determinant:__init__:160 - Using cached signs from file signs_cache/d49b2fd515a226965b09a9b91271c65e.npy
2023-04-26 14:29:43.839 | DEBUG    | __main__:<module>:66 - n_batches=422


KeyboardInterrupt: 

In [27]:
with torch.no_grad():
    prediction_signs = pd.Series(
        np.sign(det(torch.arange(0, len(det.basis.states)).long()).numpy()), index=det.basis.states
    )

In [20]:
series = fourier_expand(LBFFromEigenstateSeries(system.lattice, prediction_signs))

2023-04-26 14:43:23.795 | DEBUG    | fast_boolean_analysis:fourier_expand:245 - Finding signal
2023-04-26 14:43:24.661 | DEBUG    | fast_boolean_analysis:fourier_expand:247 - Doing 


In [21]:
series.how_many_terms_to_achieve_score(0.8, 'accuracy')

2023-04-26 14:43:48.537 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:162 - min_terms=1, max_terms=16777216
2023-04-26 14:43:49.334 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:165 - mid=8388608, score=1.0
2023-04-26 14:43:49.335 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:168 - score >= target_score, so we can decrease max_terms
2023-04-26 14:43:49.336 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:162 - min_terms=1, max_terms=8388608
2023-04-26 14:43:50.280 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:165 - mid=4194304, score=1.0
2023-04-26 14:43:50.281 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:168 - score >= target_score, so we can decrease max_terms
2023-04-26 14:43:50.282 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:162 - min_terms=1, max_terms=4194304
2023-04-26 14:43:51.089 | DEBUG    | fast_boolean_analysis:how_many_terms_to_

(True,
 45151,
 array([-0.14865971, -0.11355305, -0.10114193, ..., -0.10114193,
        -0.11355305, -0.14865971]))

In [23]:
fourier_expand(LBFFromSpinSystem(system)).how_many_terms_to_achieve_score(0.8, 'accuracy')

2023-04-26 14:45:02.890 | DEBUG    | fast_boolean_analysis:fourier_expand:245 - Finding signal
2023-04-26 14:45:02.893 | DEBUG    | heisenberg_hamiltonians:get_ground_state_in_full_basis:192 - Finding coeffs
/vol/tcm10/ischurov/frustrations-eda/heisenberg_hamiltonians.py:195: ComplexWarning: Casting complex values to real discards the imaginary part
  coeffs[self.canonical_basis.states] = self.get_ground_state_in_canonical_basis()
2023-04-26 14:45:03.308 | DEBUG    | fast_boolean_analysis:fourier_expand:247 - Doing 
2023-04-26 14:45:05.538 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:162 - min_terms=1, max_terms=16777216
2023-04-26 14:45:06.429 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:165 - mid=8388608, score=1.0
2023-04-26 14:45:06.431 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:168 - score >= target_score, so we can decrease max_terms
2023-04-26 14:45:06.432 | DEBUG    | fast_boolean_analysis:how_many_terms_to_ach

(True,
 30407,
 array([ 0.53129363,  0.10219073, -0.01356339, ..., -0.01356339,
         0.10219073,  0.53129363]))

In [25]:
system.get_df_eigenstate(0)['eigenstate_coeff']

4095        6.520253e-08
6143        1.917753e-08
7167       -1.202650e-19
7679       -1.983175e-07
7935       -2.243748e-07
                ...     
16769280   -2.243748e-07
16769536   -1.983175e-07
16770048   -1.202650e-19
16771072    1.917753e-08
16773120    6.520253e-08
Name: eigenstate_coeff, Length: 2704156, dtype: float64

In [28]:
fourier_expand(
    LBFFromEigenstateSeries(
        system.lattice, prediction_signs * np.sign(system.get_df_eigenstate(0)["eigenstate_coeff"])
    )
).how_many_terms_to_achieve_score(0.8, "accuracy")

2023-04-26 14:48:49.334 | DEBUG    | fast_boolean_analysis:fourier_expand:245 - Finding signal
2023-04-26 14:48:50.120 | DEBUG    | fast_boolean_analysis:fourier_expand:247 - Doing 
2023-04-26 14:48:51.824 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:162 - min_terms=1, max_terms=16777216
2023-04-26 14:48:52.811 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:165 - mid=8388608, score=1.0
2023-04-26 14:48:52.813 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:168 - score >= target_score, so we can decrease max_terms
2023-04-26 14:48:52.813 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:162 - min_terms=1, max_terms=8388608
2023-04-26 14:48:53.834 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:165 - mid=4194304, score=1.0
2023-04-26 14:48:53.835 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:168 - score >= target_score, so we can decrease max_terms
2023-04-26 14:48:53.836

(True,
 185448,
 array([0.16145802, 0.13881683, 0.0880518 , ..., 0.0880518 , 0.13881683,
        0.16145802]))